# Lib


In [1]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from gensim.models import Word2Vec
from node2vec import Node2Vec
from sklearn.decomposition import PCA

import warnings

warnings.filterwarnings("ignore")

b:\_projects\social-analysis-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data


In [2]:
# Load Labels
df_target = pd.read_csv("../../data/github_social_network/musae_git_target.csv")
df_target.columns = df_target.columns.str.strip().str.lower().str.replace(" ", "_")

# Load Edges
df_edges = pd.read_csv("../../data/github_social_network/musae_git_edges.csv")
df_edges.columns = df_edges.columns.str.strip().str.lower().str.replace(" ", "_")

# Load Features
with open("../../data/github_social_network/musae_git_features.json", "r") as f:
    features_json = json.load(f)

# Combination


## Merge


In [3]:
df_target.rename(columns={"id": "source"}, inplace=True)
df_target.head()

,source,name,ml_target
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


In [4]:
df_edges.rename(columns={"id_1": "source", "id_2": "target"}, inplace=True)
df_edges.head()

,source,target
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


In [5]:
df = df_edges.merge(df_target, on="source", how="left")

In [6]:
df.head()

,source,target,name,ml_target
0,0,23977,Eiryyy,0
1,1,34526,shawflying,0
2,1,2370,shawflying,0
3,1,14683,shawflying,0
4,1,29982,shawflying,0


## Feature


In [7]:
feature_ids = sorted(
    {
        int(feature_id)
        for feature_list in features_json.values()
        for feature_id in feature_list
    }
)
feature_id_set = set(feature_ids)

print(f"Total unique features: {len(feature_ids)}")
print(f"Feature id range: {min(feature_ids)} - {max(feature_ids)}")

Total unique features: 4005
Feature id range: 0 - 4004


In [8]:
valid_feature_set = set(int(fid) for fid in feature_ids)
target_nodes = df_target["source"].values

row_map = {node_id: idx for idx, node_id in enumerate(target_nodes)}
col_map = {int(fid): idx for idx, fid in enumerate(feature_ids)}
matrix_np = np.zeros((len(target_nodes), len(feature_ids)), dtype=np.uint8)

for source_id_str, feature_list in features_json.items():
    source_id = int(source_id_str)
    if source_id not in row_map:
        continue

    row_idx = row_map[source_id]

    valid_col_indices = [
        col_map[int(f_id)] for f_id in feature_list if int(f_id) in valid_feature_set
    ]

    if valid_col_indices:
        matrix_np[row_idx, valid_col_indices] = 1

feature_columns = [f"feature_{fid}" for fid in feature_ids]

feature_matrix = pd.DataFrame(matrix_np, index=target_nodes, columns=feature_columns)

In [9]:
feature_matrix.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_3995,feature_3996,feature_3997,feature_3998,feature_3999,feature_4000,feature_4001,feature_4002,feature_4003,feature_4004
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Conclusion

In [10]:
df = df.merge(feature_matrix, left_on="source", right_index=True, how="left")

In [11]:
df.head()

,source,target,name,ml_target,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,...,feature_3995,feature_3996,feature_3997,feature_3998,feature_3999,feature_4000,feature_4001,feature_4002,feature_4003,feature_4004
0,0,23977,Eiryyy,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,34526,shawflying,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,2370,shawflying,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,14683,shawflying,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,29982,shawflying,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Feature extraction


# Export


In [14]:
df.to_parquet(
    "../../data/processed/processed.parquet",
    engine="fastparquet",
    compression="snappy",
    index=False,
)